# Grammar-constrained decoding

Option 1. The model writes a signup line such as `name=maya grade=10 club=robotics`.

There are four runs. Two prompts (0-shot and 1-shot) cross two decoders (baseline and constrained). The 1-shot prompt adds one worked example. That example is not an evaluation sentence. Inside one prompt, the baseline and the constrained decoder use the same text.

Sentences are split into `standard` (one fixed phrasing) and `held_out` (other phrasings, including extra details that should be ignored). A line is correct when it parses and the name, grade, and club match the sentence.

In [1]:
import os
import warnings

os.environ["TQDM_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore", message="IProgress not found")

import demo

counts = demo.check_problems()
print("model:", demo.MODEL_NAME)
print("dtype: float32")
print("decoding: greedy (do_sample=False)")
print("max_new_tokens:", demo.MAX_NEW_TOKENS)
print("seed:", demo.SEED)
print("standard:", counts["standard"])
print("held_out:", counts["held_out"])
print("worked example:", demo.EXAMPLE_SENTENCE)
print("worked line:", demo.gold_line(demo.EXAMPLE_NAME, demo.EXAMPLE_GRADE, demo.EXAMPLE_CLUB))
print()
print("system prompt:")
print(demo.SYSTEM)
print()
print("0-shot user text:")
print(demo.user_text("<sentence>"))
print()
print("1-shot adds this earlier turn, then the same user text:")
print(demo.user_text(demo.EXAMPLE_SENTENCE))
print(demo.gold_line(demo.EXAMPLE_NAME, demo.EXAMPLE_GRADE, demo.EXAMPLE_CLUB))


model: Qwen/Qwen2.5-0.5B-Instruct
dtype: float32
decoding: greedy (do_sample=False)
max_new_tokens: 32
seed: 0
standard: 25
held_out: 25
worked example: Alex is in grade 7 and wants the music club.
worked line: name=alex grade=7 club=music

system prompt:
You write exactly one signup line and nothing else. The form is name=<word> grade=<number> club=<word>. Use lowercase letters only. The club is one word and does not include the word club. No quotes and no extra words.

0-shot user text:
Write the signup line for this sentence: <sentence>

1-shot adds this earlier turn, then the same user text:
Write the signup line for this sentence: Alex is in grade 7 and wants the music club.
name=alex grade=7 club=music


## Held fixed

Inside one prompt, both decoders use the model, that prompt, greedy decoding, `max_new_tokens`, and the seed above. The grammar mask is the only change.

The 0-shot prompt and the 1-shot prompt differ by the one worked example. The next cell runs all 50 sentences in all four conditions and writes `results.json`.

In [2]:
demo.main()


Grammar-constrained decoding
model: Qwen/Qwen2.5-0.5B-Instruct
dtype: float32
decoding: greedy (do_sample=False)
max_new_tokens: 32
seed: 0
standard sentences: 25
held_out sentences: 25
running: 50
worked example: Alex is in grade 7 and wants the music club.
worked line: name=alex grade=7 club=music
Inside one prompt, held fixed: model, prompt text, greedy decoding, max_new_tokens, seed
Only difference inside one prompt: grammar mask on or off
0-shot vs 1-shot changes the prompt by one example and nothing else
loading model...


grammar tokens: 41972

[1/50] standard: Maya is in grade 10 and wants the robotics club.
  gold: name=maya grade=10 club=robotics


  0-shot baseline     'Maya is in grade 10 and wants the robotics club.'  invalid  0.7s


  0-shot constrained  'name=maya grade=10 club=robotics'  correct  1.4s


  1-shot baseline     'name=maya grade=10 club=robotics'  correct  0.6s


  1-shot constrained  'name=maya grade=10 club=robotics'  correct  1.5s

[2/50] standard: Sam is in grade 11 and wants the chess club.
  gold: name=sam grade=11 club=chess


... (193 outputs left out)

  0-shot constrained  'name=willow grade=11 club=drama'  wrong_meaning  1.3s


  1-shot baseline     'club=dance club full where=grade 11 where=wesley where=drama club'  invalid  0.9s


  1-shot constrained  'name=wesley grade=11 club=drama'  correct  1.3s

Summary
prompt   decoding     split        n  valid  correct    wrong  invalid forced  avg_s total_s
0-shot   baseline     standard    25     0%     0/25     0/25    25/25      0    0.7    16.7
0-shot   baseline     held_out    25     0%     0/25     0/25    25/25      0    0.9    22.6
0-shot   baseline     overall     50     0%     0/50     0/50    50/50      0    0.8    39.3
0-shot   constrained  standard    25   100%    16/25     9/25     0/25      0    1.3    32.0
0-shot   constrained  held_out    25   100%    12/25    13/25     0/25      0    1.4    35.0
0-shot   constrained  overall     50   100%    28/50    22/50     0/50      0    1.3    67.0
1-shot   baseline     standard    25   100%    22/25     3/25     0/25      0    0.6    14.7
1-shot   baseline     held_out    25    48%    11/25     1/25    13/25      0    0.7    17.6
1-shot   baseline     overall     50    74%    33/50     4/50    13/50      0    0.

[{'prompt': '0-shot',
  'decoding': 'baseline',
  'split': 'standard',
  'n': 25,
  'correct': 0,
  'wrong_meaning': 0,
  'invalid': 25,
  'syntactic_validity': 0.0,
  'correct_rate': 0.0,
  'wrong_meaning_rate': 0.0,
  'invalid_rate': 1.0,
  'force_finish': 0,
  'average_seconds': 0.668284,
  'total_seconds': 16.7071},
 {'prompt': '0-shot',
  'decoding': 'baseline',
  'split': 'held_out',
  'n': 25,
  'correct': 0,
  'wrong_meaning': 0,
  'invalid': 25,
  'syntactic_validity': 0.0,
  'correct_rate': 0.0,
  'wrong_meaning_rate': 0.0,
  'invalid_rate': 1.0,
  'force_finish': 0,
  'average_seconds': 0.9024880000000001,
  'total_seconds': 22.5622},
 {'prompt': '0-shot',
  'decoding': 'baseline',
  'split': 'overall',
  'n': 50,
  'correct': 0,
  'wrong_meaning': 0,
  'invalid': 50,
  'syntactic_validity': 0.0,
  'correct_rate': 0.0,
  'wrong_meaning_rate': 0.0,
  'invalid_rate': 1.0,
  'force_finish': 0,
  'average_seconds': 0.785386,
  'total_seconds': 39.2693},
 {'prompt': '0-shot',
  '